# NHANES 2021–2023 — Exploratory Data Analysis
### Model 2: OW_I vs OW_II Binary Classifier

**Objective:** This notebook prepares and explores the NHANES 2021–2023 dataset to support training Model 2 — a binary classifier that distinguishes Overweight Class I (BMI 25–29.9) from Overweight Class II (BMI 30+). Model 2 is triggered by the router when Model 1 produces ambiguous predictions in the overweight range.

**Dataset:** CDC NHANES 2021–2023 cycle (`_L` suffix files). Covers 11,933 participants across demographics, physical exam, laboratory, and questionnaire data.

---

**Notebook structure:**
1. Raw file inspection
2. Merge & integrity check
3. Sentinel value cleaning
4. Smart summary of all 256 columns
5. Missingness audit by domain
6. Model 2 feature statistics
7. Feature correlation analysis
8. Target variable definition & class balance

---
## 1. Raw File Inspection

Before merging anything, we inspect each raw XPT file individually. This gives us a baseline understanding of what each file contains, how many participants it covers, and what the raw values look like before any cleaning.

NHANES distributes data across separate files by domain (demographics, exam, lab, questionnaire). Each file links back to a participant via `SEQN` — the unique participant ID. Not every participant appears in every file, because some exams (e.g. fasting labs) are only conducted on a subsample.

In [ ]:
import pandas as pd
import os
import glob

folders = ['demographics_data', 'exam_data', 'lab_data', 'quest_data']

for folder in folders:
    files = glob.glob(os.path.join(folder, '*.xpt')) + glob.glob(os.path.join(folder, '*.XPT'))
    
    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"\n" + "="*50)
        print(f"FILE: {file_name} (in {folder})")
        print("="*50)
        
        try:
            df = pd.read_sas(file_path)
            df.columns = df.columns.str.upper()
            
            print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
            print("\n--- Column Names (Variables) ---")
            print(list(df.columns))
            print("\n--- Data Sample ---")
            display(df.head())
            print("\n--- Basic Statistics (Numerical) ---")
            display(df.describe())
            
        except Exception as e:
            print(f"Could not read {file_name}: {e}")
            

---
## 2. Merge All Files into a Master DataFrame

We merge all domain files into a single master dataframe using `SEQN` as the join key.

**Key decisions:**
- **Anchor on `DEMO_L`** — Demographics has all 11,933 participants. Using it as the base with `how='left'` ensures we never lose a participant during the merge. Lab and exam files will introduce NaNs for participants not in those subsamples, which is expected and informative.
- **Conflict resolution** — Some weight columns (e.g. `WTPH2YR`, `WTSAF2YR`) appear in multiple files. We resolve these by filling from the master first, rather than creating `_x`/`_y` duplicate columns.
- **Deduplication** — Each file is deduplicated on `SEQN` before merging to prevent row explosion from any unexpected duplicates in the source files.

After merging, we save four subsets based on biomarker availability, because fasting glucose and insulin are only available for the ~3,500-person fasting subsample. Model 2 training will use this subset.

In [ ]:
output_dir = 'merged_data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

folders = ['demographics_data', 'exam_data', 'lab_data', 'quest_data']
all_dfs = []

print("--- Step 1: Loading raw XPT files ---")
for folder in folders:
    files = glob.glob(os.path.join(folder, '*.xpt')) + glob.glob(os.path.join(folder, '*.XPT'))
    for file_path in files:
        try:
            temp_df = pd.read_sas(file_path)
            temp_df.columns = temp_df.columns.str.upper()
            temp_df = temp_df.drop_duplicates(subset=['SEQN'])
            all_dfs.append({'name': os.path.basename(file_path), 'df': temp_df})
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

# Anchor merge on DEMO_L
base_entry = next((item for item in all_dfs if 'DEMO_L' in item['name']), all_dfs[0])
master_df = base_entry['df']

print(f"\n--- Step 2: Merging files onto {base_entry['name']} ---")

for entry in all_dfs:
    if entry['name'] == base_entry['name']:
        continue
    
    new_df = entry['df']
    conflicts = set(master_df.columns) & set(new_df.columns) - {'SEQN'}
    
    if conflicts:
        print(f"Resolving conflicts in {entry['name']}: {list(conflicts)}")
        for col in conflicts:
            master_df[col] = master_df[col].fillna(
                master_df['SEQN'].map(new_df.set_index('SEQN')[col])
            )
        new_df = new_df.drop(columns=list(conflicts))
    
    master_df = pd.merge(master_df, new_df, on='SEQN', how='left')

# Save master
master_path = os.path.join(output_dir, 'master_merged_data.csv')
master_df.to_csv(master_path, index=False)
print(f"\nMaster file saved: {master_path} | Columns: {len(master_df.columns)}")

# Save biomarker subsets
print("\n--- Step 3: Saving Subsets into 'merged_data/' ---")

import numpy as np

subsets = {
    "nhanes_glucose_only.csv":          master_df[master_df['LBXGLU'].notna()].copy(),
    "nhanes_insulin_only.csv":          master_df[master_df['LBXIN'].notna()].copy(),
    "nhanes_complete_biomarkers.csv":   master_df[master_df['LBXGLU'].notna() & master_df['LBXIN'].notna()].copy(),
    "nhanes_no_biomarkers.csv":         master_df[master_df['LBXGLU'].isna() & master_df['LBXIN'].isna()].copy()
}

for filename, data in subsets.items():
    path = os.path.join(output_dir, filename)
    data.to_csv(path, index=False)
    print(f"Saved {path:40} | Rows: {len(data):<6}")

print("\nAll merges complete.")

---
## 3. Integrity Check

Before proceeding, we verify the merge produced clean output. Three checks:

1. **Column preservation** — All 256 original variables from the source XPT files should appear in the master. No columns should have been silently dropped, and no `_x`/`_y` duplicate columns should exist.
2. **Row uniqueness** — Each `SEQN` should appear exactly once. If a participant appears twice, it means a merge went wrong (a many-to-one join error caused row explosion).
3. **Biomarker sanity check** — Glucose and insulin values in the complete biomarker subset should fall within physiologically plausible ranges. Average fasting glucose should be between 60–200 mg/dL. Average insulin should be between 2–30 uIU/mL. Survey weights should be non-null.

In [ ]:
print("--- Integrity Check: Column Preservation ---")

all_original_vars = set()
for folder in folders:
    files = glob.glob(os.path.join(folder, '*.xpt')) + glob.glob(os.path.join(folder, '*.XPT'))
    for file_path in files:
        try:
            with pd.read_sas(file_path, chunksize=1) as reader:
                df_header = next(reader)
                all_original_vars.update([c.upper() for c in df_header.columns])
        except:
            continue

df_master = pd.read_csv(master_path, nrows=0)
merged_vars = set(df_master.columns)

missing = all_original_vars - merged_vars
duplicates = [c for c in df_master.columns if '_X' in c or '_Y' in c or '_DUP' in c]

print(f"Original unique variables found: {len(all_original_vars)}")
print(f"Master file variables found:     {len(merged_vars)}")

if not missing and not duplicates:
    print("✅ SUCCESS: 1:1 Column Integrity. No variables lost, no duplicates created.")
else:
    if missing:     print(f"⚠️  MISSING from merge: {missing}")
    if duplicates:  print(f"⚠️  DUPLICATE columns found: {duplicates}")

print("\n--- Integrity Check: Row Uniqueness ---")
df_full = pd.read_csv(master_path, usecols=['SEQN'])
if df_full['SEQN'].is_unique:
    print(f"✅ SUCCESS: Every participant (SEQN) is unique. No row explosion occurred.")
else:
    print(f"❌ FAILURE: Duplicate SEQN found. Merge logic is flawed.")

print("\n--- Integrity Check: Biomarker Sanity ---")
df_bio = pd.read_csv(
    os.path.join(output_dir, 'nhanes_complete_biomarkers.csv'),
    usecols=['SEQN', 'LBXGLU', 'LBXIN', 'WTSAF2YR']
)

avg_glu      = df_bio['LBXGLU'].mean()
avg_ins      = df_bio['LBXIN'].mean()
null_weight  = df_bio['WTSAF2YR'].isna().sum()

print(f"\nBiomarker Verification (Sample Size: {len(df_bio)}):")
print(f" - Average Glucose: {avg_glu:.2f} mg/dL  {'✅' if 60 < avg_glu < 200 else '❌'}")
print(f" - Average Insulin: {avg_ins:.2f} uIU/mL {'✅' if 2 < avg_ins < 30 else '❌'}")
print(f" - Missing Weights: {null_weight}          {'✅' if null_weight == 0 else '⚠️  Check Weights'}")

if null_weight == 0 and 60 < avg_glu < 200:
    print("\nOVERALL STATUS: Data is CLEAN and ready for analysis.")

---
## 4. Load Master & Clean Sentinel Values

We load the master CSV and handle two types of problematic values before any analysis:

**SAS system missing (5.397605e-79):** When SAS exports XPT files, its internal missing value sentinel becomes this near-zero float. It is not a real measurement — it must be converted to `NaN`. We catch it by zeroing out any float value smaller than 1e-70.

**NHANES refusal/don't-know codes:** Questionnaire columns encode participant refusals as 7, 77, 777 and don't-know responses as 9, 99, 999, 9999. These look like real numbers but carry no clinical meaning. We replace them with `NaN` in questionnaire columns only — we do not apply this to lab columns, where values like 99 mg/dL are a legitimate glucose reading.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('merged_data/master_merged_data.csv')
print(f"Master dataframe loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# --- Clean SAS sentinel values ---
# 5.397605e-79 is the SAS internal missing value that survived XPT export.
# Any float this close to zero is a sentinel, not a real measurement.
sentinel = 5.397605e-79
for col in df.select_dtypes(include='float').columns:
    df[col] = df[col].replace(sentinel, np.nan)
    df.loc[df[col].abs() < 1e-70, col] = np.nan

print("✅ SAS sentinel values (5.397605e-79) replaced with NaN.")

# --- Clean NHANES refusal/don't-know codes (questionnaire columns only) ---
# These codes are safe to replace in questionnaire variables.
# We deliberately skip lab columns where these numbers are valid readings.
nhanes_missing_codes = [7, 9, 77, 99, 777, 999, 9999]

quest_cols = [
    'DIQ010', 'MCQ160A', 'MCQ160B', 'MCQ160C', 'MCQ160D',
    'MCQ160E', 'MCQ160F', 'BPQ020', 'SMQ020', 'ALQ111',
    'PAD680', 'SLD012', 'WHD010', 'WHD020', 'KIQ022',
    'HSQ590', 'SLQ300', 'SLQ320'
]

for col in quest_cols:
    if col in df.columns:
        df[col] = df[col].replace(nhanes_missing_codes, np.nan)

print("✅ NHANES refusal/don't-know codes replaced with NaN in questionnaire columns.")
print(f"\nFinal dataframe shape: {df.shape}")

---
## 5. Smart Summary — All 256 Columns

With 256 columns, a standard `df.describe()` is unreadable. Instead we build a compact summary that surfaces what actually matters:

- `null_pct` — sorted descending, so the most unusable columns appear at the top immediately. Columns above 80% missing are strong candidates for exclusion.
- `unique` — helps distinguish binary flags (2 values), ordinal codes (3–10 values), and continuous measurements (many values).
- `mean / std / min / max` — quick sanity check for implausible values that survived cleaning.

We also export this to CSV since 256 rows is too many to comfortably scan in a notebook.

In [ ]:
def smart_summary(df):
    summary = pd.DataFrame({
        'dtype':    df.dtypes,
        'non_null': df.notna().sum(),
        'null_pct': (df.isna().mean() * 100).round(1),
        'unique':   df.nunique(),
        'mean':     df.mean(numeric_only=True),
        'std':      df.std(numeric_only=True),
        'min':      df.min(numeric_only=True),
        'max':      df.max(numeric_only=True),
    })
    return summary.sort_values('null_pct', ascending=False)

summary = smart_summary(df)
display(summary)

# Export — reading 256 rows inline is impractical
summary.to_csv('merged_data/column_summary.csv')
print("\nFull summary exported to merged_data/column_summary.csv")

---
## 6. Missingness Audit by Domain

We visualise missingness as a heatmap, with columns grouped by their source domain. Each column is a vertical stripe — white means the value is present, blue means it is missing for that participant.

**Why grouped by domain?** Missingness patterns in NHANES are structurally driven, not random:
- **Demographics** — should be nearly complete. Heavy missingness here signals household-reference-person variables (not the participant themselves).
- **Exam** — missingness follows exam eligibility. BAX (balance) was only given to a subset. BMX and BPXO should be mostly present.
- **Labs** — the fasting subsample creates a clear horizontal band. Glucose and insulin are missing for ~65% of participants by design.
- **Questionnaire** — skip patterns create column-level missingness. A question only asked of diabetics will be missing for everyone else.

Understanding *why* something is missing is critical before deciding how to handle it in the model.

In [ ]:
# Group columns by domain prefix
demo_cols  = [c for c in df.columns if c.startswith(('SEQN','RID','DMD','WTINT','WTMEC','SDM','IND'))]
exam_cols  = [c for c in df.columns if c.startswith(('BMX','BMI','BPX','BAX','BAQ'))]
lab_cols   = [c for c in df.columns if c.startswith(('LBX','LBD','WTS'))]
quest_cols = [c for c in df.columns if c.startswith(('MCQ','DIQ','BPQ','ALQ','SMQ','PAD','SLQ','WHD','KIQ','HSQ'))]

fig, axes = plt.subplots(4, 1, figsize=(22, 28))
domains = [
    ('Demographics', demo_cols),
    ('Exam',         exam_cols),
    ('Labs',         lab_cols),
    ('Questionnaire',quest_cols)
]

for ax, (title, cols) in zip(axes, domains):
    cols_present = [c for c in cols if c in df.columns]
    sns.heatmap(
        df[cols_present].isna(),
        ax=ax, cbar=False, yticklabels=False,
        cmap='Blues', xticklabels=True
    )
    ax.set_title(f'Missing Data Audit: {title} (Blue = Missing)', fontsize=13, pad=10)
    ax.tick_params(axis='x', rotation=90, labelsize=7)

plt.tight_layout(pad=3.0)
plt.savefig('merged_data/missingness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to merged_data/missingness_heatmap.png")

---
## 7. Model 2 Feature Statistics

We now focus exclusively on the 16 features selected for Model 2. These were chosen because they are clinically meaningful for distinguishing OW_I from OW_II, and they are available as direct physical or lab measurements — not self-reported.

The feature set covers four clinical domains:
- **Anthropometrics:** BMI, weight, height, waist circumference — the core body composition signals
- **Cardiovascular:** Three systolic and three diastolic readings — taken in sequence to capture the white-coat effect decay
- **Metabolic labs:** Total cholesterol, fasting glucose, fasting insulin, HbA1c
- **Sociodemographic:** Age and income-to-poverty ratio — both correlate with metabolic risk independent of body size

Note the different sample sizes: BMI has ~8,471 observations while glucose has only ~3,672, because glucose requires fasting. This missingness gap directly determines our Model 2 training set size.

In [ ]:
m2_features = [
    'BMXBMI',   # Body Mass Index — the primary classification variable
    'BMXWT',    # Weight (kg)
    'BMXHT',    # Height (cm)
    'BMXWAIST', # Waist circumference — strongest predictor of visceral fat beyond BMI
    'BPXOSY1',  # Systolic BP — reading 1 (typically highest due to white-coat effect)
    'BPXOSY2',  # Systolic BP — reading 2
    'BPXOSY3',  # Systolic BP — reading 3 (most stable)
    'BPXODI1',  # Diastolic BP — reading 1
    'BPXODI2',  # Diastolic BP — reading 2
    'BPXODI3',  # Diastolic BP — reading 3
    'LBXTC',    # Total cholesterol (mg/dL)
    'LBXGLU',   # Fasting glucose (mg/dL) — fasting subsample only (~3,600 participants)
    'LBXIN',    # Fasting insulin (uIU/mL) — fasting subsample only
    'LBXGH',    # Glycohemoglobin / HbA1c (%) — 3-month blood sugar average
    'RIDAGEYR', # Age in years
    'INDFMPIR'  # Income-to-poverty ratio (0–5 scale; 5 = highest income bracket)
]

m2_features = [c for c in m2_features if c in df.columns]

print(f"Model 2 features available: {len(m2_features)} / 16")
display(df[m2_features].describe().T.round(2))

---
## 8. Feature Correlation Analysis

We compute pairwise Pearson correlations across all Model 2 features and visualise the lower triangle only (the upper triangle is a mirror image).

**What to look for:**
- **High correlations (>0.85) between features** signal multicollinearity. For tree-based models (XGBoost, LightGBM) this is not a problem — the model will just pick the more informative one. For linear models it would need to be addressed.
- **Three BP readings per type** are expected to correlate ~0.93–0.94 with each other. This confirms we should average them into a single `SYS_BP` and `DIA_BP` feature before modeling.
- **BMI / Weight / Waist** cluster together at 0.91–0.94. All three stay in the model because they capture slightly different aspects of body composition.
- **Glucose and insulin at 0.21** — low correlation despite both being metabolic markers. They measure different things: glucose is the circulating level, insulin is the hormonal response. Keeping both is justified.
- **Age correlating with everything** is expected. Older participants tend to have higher BP, worse metabolic markers, and different body composition.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df[m2_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, ax=ax,
    annot_kws={'size': 8}, linewidths=0.5
)
ax.set_title('Model 2 Feature Correlations')
plt.tight_layout()
plt.savefig('merged_data/m2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to merged_data/m2_correlation.png")

---
## 9. Target Variable Definition & Class Balance

We define the Model 2 target variable using WHO/CDC BMI classification thresholds:
- **OW_I** — BMI 25.0 to 29.9 (Overweight Class I)
- **OW_II** — BMI 30.0 and above (Overweight Class II / Obese)

Participants with BMI below 25 are excluded from Model 2 — they are classified by Model 1 and will never trigger the router.

**Why check class balance?** An imbalanced dataset means the model can get a high accuracy score by simply predicting the majority class every time. We need to know the split before choosing a training strategy. A 55/45 split is mild enough that we don't need SMOTE — using `class_weight='balanced'` in sklearn or `scale_pos_weight` in XGBoost is sufficient.

**Usable rows for training:** We further filter to participants who have *both* a BMI classification *and* fasting glucose. This is the realistic training set ceiling for the full clinical feature set.

In [ ]:
# Define OW_I / OW_II using WHO BMI cutoffs
# right=False means the left boundary is inclusive: [25, 30) and [30, inf)
df['OW_CLASS'] = pd.cut(
    df['BMXBMI'],
    bins=[25, 30, np.inf],
    labels=['OW_I', 'OW_II'],
    right=False
)

print("=" * 40)
print("Full overweight population (all participants with BMI >= 25):")
print(df['OW_CLASS'].value_counts())
print()
print(df['OW_CLASS'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

print("\n" + "=" * 40)
print("Model 2 training set (BMI class + fasting glucose available):")
usable = df[df['OW_CLASS'].notna() & df['LBXGLU'].notna()]
print(f"Usable rows: {len(usable):,}")
print(usable['OW_CLASS'].value_counts())
print()
print(usable['OW_CLASS'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

print("\n" + "=" * 40)
print("Class imbalance assessment:")
ratio = usable['OW_CLASS'].value_counts(normalize=True)
imbalance = ratio.max() / ratio.min()
print(f"Majority/minority ratio: {imbalance:.2f}x")
if imbalance < 1.5:
    print("✅ Mild imbalance. Use class_weight='balanced' — SMOTE not required.")
elif imbalance < 3.0:
    print("⚠️  Moderate imbalance. Consider SMOTE or class weighting.")
else:
    print("❌ Severe imbalance. SMOTE or oversampling strongly recommended.")

---
## Summary & Next Steps

**What this EDA established:**

- The merge is clean: 256 columns, 11,933 unique participants, no row explosion, no duplicate columns.
- SAS sentinel values and NHANES refusal codes have been cleaned out.
- The fasting subsample caps our Model 2 training set at ~2,472 rows (participants with BMI >= 25 AND fasting glucose available).
- Class balance is mild (55% OW_II / 45% OW_I) — no aggressive resampling needed.
- The three BP readings per type are redundant (r = 0.93–0.94) and should be averaged before modeling.
- BMI, weight, and waist are highly correlated but all stay in — tree models handle this fine.
- Glucose and insulin are low-correlated (r = 0.21) and both earn their place as features.

**Next steps (Model 2 training notebook):**
1. Average BP readings → `SYS_BP`, `DIA_BP`
2. Create `GLU_AVAILABLE` binary flag (1 if fasting glucose present, 0 if not)
3. Filter to `RIDAGEYR >= 18` — BMI classification works differently for children
4. Train/test split stratified on `OW_CLASS`
5. Train XGBoost binary classifier with `scale_pos_weight` to handle mild imbalance
6. Evaluate with AUC-ROC, precision-recall curve, and confusion matrix

---
# Looking at imbalances

In [ ]:
df = pd.read_csv('merged_data/master_merged_data.csv')

# Clean sentinel
for col in df.select_dtypes(include='float').columns:
    df.loc[df[col].abs() < 1e-70, col] = np.nan

# Define target
df['OW_CLASS'] = pd.cut(df['BMXBMI'], bins=[25, 30, np.inf],
                         labels=['OW_I', 'OW_II'], right=False)

# Working subset — OW participants only
ow = df[df['OW_CLASS'].notna()].copy()

# ── Decode raw codes into readable labels ──────────────────────────
ow['SEX']       = ow['RIAGENDR'].map({1: 'Male', 2: 'Female'})
ow['RACE']      = ow['RIDRETH3'].map({
    1: 'Mexican American', 2: 'Other Hispanic',
    3: 'Non-Hispanic White', 4: 'Non-Hispanic Black',
    6: 'Non-Hispanic Asian', 7: 'Other/Multi'
})
ow['AGE_BIN']   = pd.cut(ow['RIDAGEYR'],
                          bins=[18, 30, 45, 60, 80],
                          labels=['18–29', '30–44', '45–59', '60–80'])
ow['PIR_BIN']   = pd.cut(ow['INDFMPIR'],
                          bins=[0, 1.3, 3.5, 5.0],
                          labels=['Low income', 'Middle income', 'High income'])
ow['BMI_SRC']   = ow['BMXBMI']   # keep raw for distribution plots
ow['WAIST_SRC'] = ow['BMXWAIST']

# ── Helper ─────────────────────────────────────────────────────────
def balance_table(df, col):
    """Count + % per OW class for a categorical variable."""
    ct = df.groupby([col, 'OW_CLASS'], observed=True).size().unstack(fill_value=0)
    ct['Total'] = ct.sum(axis=1)
    for c in ['OW_I', 'OW_II']:
        ct[f'{c} %'] = (ct[c] / ct['Total'] * 100).round(1)
    return ct

# ══════════════════════════════════════════════════════════════════
# FIGURE 1 — Categorical imbalance checks (2×2 grid)
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Class Distribution Across Demographic Groups', fontsize=14, y=1.01)

categorical_checks = [
    ('SEX',     'Sex'),
    ('RACE',    'Race / Ethnicity'),
    ('AGE_BIN', 'Age Group'),
    ('PIR_BIN', 'Income Level'),
]

for ax, (col, title) in zip(axes.flat, categorical_checks):
    sub = ow[[col, 'OW_CLASS']].dropna()
    ct = sub.groupby([col, 'OW_CLASS'], observed=True).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

    ct_pct.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'],
                edgecolor='white', width=0.65)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('% within group')
    ax.set_xlabel('')
    ax.set_ylim(0, 100)
    ax.axhline(50, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Class', fontsize=8)

    # Print balance table to console
    print(f"\n── {title} ──")
    print(balance_table(ow, col).to_string())

plt.tight_layout()
plt.savefig('merged_data/imbalance_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════════
# FIGURE 2 — Continuous feature distributions by class
# ══════════════════════════════════════════════════════════════════
continuous_checks = [
    ('RIDAGEYR',  'Age (years)'),
    ('BMXWAIST',  'Waist Circumference (cm)'),
    ('BPXOSY1',   'Systolic BP — reading 1 (mmHg)'),
    ('LBXTC',     'Total Cholesterol (mg/dL)'),
    ('LBXGLU',    'Fasting Glucose (mg/dL)'),
    ('INDFMPIR',  'Income-to-Poverty Ratio'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Continuous Feature Distributions — OW_I vs OW_II', fontsize=14, y=1.01)

for ax, (col, label) in zip(axes.flat, continuous_checks):
    sub = ow[[col, 'OW_CLASS']].dropna()
    for cls, color in [('OW_I', '#4C72B0'), ('OW_II', '#DD8452')]:
        vals = sub[sub['OW_CLASS'] == cls][col]
        ax.hist(vals, bins=35, alpha=0.55, color=color, label=cls, density=True)

    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

    # Print summary stats
    print(f"\n── {label} ──")
    print(sub.groupby('OW_CLASS', observed=True)[col]
            .agg(['mean','median','std','min','max'])
            .round(2).to_string())

plt.tight_layout()
plt.savefig('merged_data/imbalance_continuous.png', dpi=150, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════════
# FIGURE 3 — Fasting subsample bias check
# Are the people WITH glucose data representative of the full OW group?
# ══════════════════════════════════════════════════════════════════
ow['HAS_GLUCOSE'] = ow['LBXGLU'].notna().map({True: 'Has glucose', False: 'No glucose'})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Fasting Subsample Bias Check\n(Are glucose-available participants representative?)',
             fontsize=12, y=1.02)

for ax, (col, title) in zip(axes, [('SEX', 'Sex'), ('AGE_BIN', 'Age Group'), ('RACE', 'Race')]):
    sub = ow[[col, 'HAS_GLUCOSE']].dropna()
    ct = sub.groupby([col, 'HAS_GLUCOSE'], observed=True).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct[['Has glucose']].plot(kind='bar', ax=ax, color='#2ca02c',
                                  edgecolor='white', legend=False)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('% with glucose data')
    ax.set_ylim(0, 100)
    ax.axhline(ct_pct['Has glucose'].mean(), color='red',
               linestyle='--', linewidth=1, label=f"avg {ct_pct['Has glucose'].mean():.0f}%")
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('merged_data/fasting_subsample_bias.png', dpi=150, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════════
# SUMMARY TABLE — overall imbalance snapshot
# ══════════════════════════════════════════════════════════════════
print("\n" + "="*50)
print("IMBALANCE SUMMARY")
print("="*50)
for col, title in categorical_checks:
    sub = ow[[col, 'OW_CLASS']].dropna()
    ct = sub.groupby([col, 'OW_CLASS'], observed=True).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    spread = ct_pct['OW_II'].max() - ct_pct['OW_II'].min()
    flag = '⚠️ ' if spread > 15 else '✅'
    print(f"{flag} {title:25} — OW_II % range: "
          f"{ct_pct['OW_II'].min():.0f}% – {ct_pct['OW_II'].max():.0f}%  "
          f"(spread: {spread:.0f}pp)")

---
# Time to decode and encode the column values

In [ ]:
import json

df = pd.read_csv('merged_data/master_merged_data.csv')

# Clean sentinel
for col in df.select_dtypes(include='float').columns:
    df.loc[df[col].abs() < 1e-70, col] = np.nan

output = {}

for col in df.columns:
    series = df[col].dropna()
    unique_vals = sorted(series.unique().tolist())
    
    output[col] = {
        'dtype':       str(df[col].dtype),
        'non_null':    int(series.count()),
        'null_count':  int(df[col].isna().sum()),
        'null_pct':    round(df[col].isna().mean() * 100, 1),
        'n_unique':    int(df[col].nunique()),
        'unique_vals': unique_vals if len(unique_vals) <= 50 else unique_vals[:50] + ['... TRUNCATED'],
    }

with open('merged_data/unique_values_audit.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Saved: merged_data/unique_values_audit.json")
print(f"Columns audited: {len(output)}")

# Also print a quick console summary so you can spot problems immediately
print("\nColumns with 50+ unique values (likely continuous):")
for col, info in output.items():
    if info['n_unique'] >= 50:
        print(f"  {col:20} — {info['n_unique']} unique")

print("\nColumns with 2–15 unique values (likely categorical/ordinal):")
for col, info in output.items():
    if 2 <= info['n_unique'] <= 15:
        print(f"  {col:20} — {info['n_unique']} unique  {info['unique_vals']}")

print("\nColumns with exactly 1 unique value (useless, drop these):")
for col, info in output.items():
    if info['n_unique'] == 1:
        print(f"  {col:20} — value: {info['unique_vals']}")